# 04 — Evaluation Dashboard

In [ ]:
import sys
import numpy as np
import pandas as pd
sys.path.insert(0, "../src")

from nba_shot_quality.data import load_or_fetch
from nba_shot_quality.features import build_feature_matrix, FEATURE_COLS
from nba_shot_quality.model import ShotQualityModel
from nba_shot_quality.evaluate import (
    plot_confusion_matrix, plot_roc_auc, plot_feature_importance,
    plot_calibration_curve, plot_team_comparison,
)
from sklearn.model_selection import train_test_split

shots = load_or_fetch(cache_dir="../data")
X, y = build_feature_matrix(shots)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = ShotQualityModel()
model.load("../models/shot_quality_model.json")
print("Model loaded.")

In [ ]:
y_proba = model._model.predict_proba(X_test[FEATURE_COLS].values)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
plot_confusion_matrix(y_test, y_pred)

In [ ]:
plot_roc_auc(y_test, y_proba)

In [ ]:
plot_feature_importance(model)

In [ ]:
plot_calibration_curve(y_test, y_proba)

In [ ]:
shots_test = X_test.copy()
shots_test["SHOT_MADE_FLAG"] = y_test.values
shots_test["TEAM_NAME"] = shots.loc[X_test.index, "TEAM_NAME"].values
plot_team_comparison(shots_test, model)